# Integration Test Demo Notebook

This notebook demonstrates basic SDG Hub functionality for integration testing purposes.

In [ ]:
# Parameters cell - papermill will inject parameters here
api_key = 'default-key'
api_base = 'http://localhost:8000/v1'
model_name = 'test-model'
test_mode = True
sample_size = 3

In [ ]:
# Standard imports
from datasets import Dataset
from openai import OpenAI

print(f'Running in test mode: {test_mode}')
print(f'Using API: {api_base}')
print(f'Sample size: {sample_size}')

In [ ]:
# Create test dataset
test_data = [
    {'text': 'Apple stock rises on earnings news', 'expected': 'Business'},
    {'text': 'New AI model breakthrough announced', 'expected': 'Sci/Tech'}, 
    {'text': 'Soccer world cup final scheduled', 'expected': 'Sports'},
    {'text': 'Election results impact markets', 'expected': 'World'}
]

dataset = Dataset.from_list(test_data[:sample_size])
print(f'Created dataset with {len(dataset)} samples')

In [ ]:
# Initialize OpenAI client
client = OpenAI(api_key=api_key, base_url=api_base)

# Test client connection
try:
    models = client.models.list()
    available_model = models.data[0].id if models.data else model_name
    print(f'Using model: {available_model}')
except Exception as e:
    print(f'Client initialization note: {e}')
    available_model = model_name

In [ ]:
# Simple classification using OpenAI API
results = []

for item in dataset:
    try:
        response = client.chat.completions.create(
            model=available_model,
            messages=[
                {'role': 'system', 'content': 'Classify the following text into one of: Business, Sci/Tech, Sports, World'},
                {'role': 'user', 'content': item['text']}
            ],
            max_tokens=10,
            temperature=0
        )
        
        prediction = response.choices[0].message.content.strip()
        results.append({
            'text': item['text'],
            'expected': item['expected'],
            'predicted': prediction
        })
        print(f'Text: {item["text"][:50]}... -> {prediction}')
        
    except Exception as e:
        print(f'Error processing item: {e}')
        results.append({
            'text': item['text'],
            'expected': item['expected'], 
            'predicted': 'ERROR'
        })

print(f'\nProcessed {len(results)} items')

In [ ]:
# Output results for validation
final_results = {
    'total_samples': len(results),
    'successful_predictions': len([r for r in results if r['predicted'] != 'ERROR']),
    'results': results,
    'test_mode': test_mode
}

print('Final Results:')
for key, value in final_results.items():
    if key != 'results':
        print(f'{key}: {value}')

# Store results for test validation
globals()['test_results'] = final_results